# DS-RxNet revision experiments (Kaggle, GPU T4 ×2)

**Before running:**
1. *Settings → Accelerator → GPU T4 ×2*, and *Settings → Internet → On* (needed for pretrained weights and the official FF++ split files).
2. *Add Input*: your FaceForensics++ frames dataset, and a Celeb-DF (v2) dataset (the one with `List_of_testing_videos.txt`).
3. Click **Save Version → Save & Run All (Commit)**. A session runs up to 12 h; the notebook pauses cleanly before the limit.

**Next sessions (until the report says ALL RUNS COMPLETE):** open the finished version's *Output*, click *New Dataset* (or add this notebook's output as an input), set `PREVIOUS_OUTPUT` in Cell 1 to that path, and Save & Run All again. Finished runs are skipped and an interrupted run resumes from its last epoch.

**What to send back:** the output of every cell that prints `PROBLEM`, and the full output of the last cell (the REPORT).

In [1]:
# ===================== CELL 1 — CONFIGURATION (only this cell should ever need editing) =====================
FFPP_DIR = None          # None = auto-detect under /kaggle/input, or e.g. '/kaggle/input/my-ffpp-frames'
CELEBDF_DIR = None       # None = auto-detect the folder that contains List_of_testing_videos.txt
PREVIOUS_OUTPUT = None   # e.g. '/kaggle/input/dsrxnet-revision-output/revision_out' from the previous session
FRAMES_PER_VIDEO = None  # None = use every extracted frame; e.g. 16 halves training time (applies to all models alike)
SEEDS = [0, 1, 2, 3, 4]
MAX_EPOCHS, PATIENCE, BATCH = 20, 4, 32
TIME_BUDGET_H = 11.0     # workers pause before Kaggle's 12-hour limit
CELEBDF_FRAMES = 32      # frames sampled per Celeb-DF test video
PRETRAINED = True

# ---------------------------------------------------------------------------------------------------------------
import os, sys, json, glob, re, time, shutil, random, subprocess
T_START = time.time()
os.system('pip install -q -U timm fvcore > /dev/null 2>&1')
os.system('pip install -q facenet-pytorch --no-deps > /dev/null 2>&1')
import numpy as np, pandas as pd, torch
WORK, CACHE, CACHE_SIZE = '/kaggle/working/revision_out', '/tmp/dsrx_cache', 256
os.makedirs(WORK, exist_ok=True); os.makedirs(CACHE, exist_ok=True)
if PREVIOUS_OUTPUT:
    for d in ['preds', 'history', 'done', 'ckpt', 'celebdf_faces']:
        if os.path.isdir(f'{PREVIOUS_OUTPUT}/{d}'): shutil.copytree(f'{PREVIOUS_OUTPUT}/{d}', f'{WORK}/{d}', dirs_exist_ok=True)
    for f in glob.glob(f'{PREVIOUS_OUTPUT}/*.json') + glob.glob(f'{PREVIOUS_OUTPUT}/*.csv'): shutil.copy(f, WORK)
shutil.rmtree(f'{WORK}/locks', ignore_errors=True)
DIAG = json.load(open(f'{WORK}/diagnostics.json')) if os.path.exists(f'{WORK}/diagnostics.json') else {}
def save_diag(): json.dump(DIAG, open(f'{WORK}/diagnostics.json', 'w'), indent=1, default=str)
import timm
print('torch', torch.__version__, '| timm', timm.__version__, '| GPUs:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
print('CPU cores:', os.cpu_count(), '| free disk /tmp: %.0f GB' % (shutil.disk_usage('/tmp').free / 1e9),
      '| /kaggle/working: %.0f GB' % (shutil.disk_usage('/kaggle/working').free / 1e9))
print('previous output:', PREVIOUS_OUTPUT, '| finished runs carried over:', len(glob.glob(f'{WORK}/done/*.json')))
if torch.cuda.device_count() < 2: print('PROBLEM: fewer than 2 GPUs — select GPU T4 x2 in Settings → Accelerator')
DIAG['env'] = dict(torch=torch.__version__, timm=timm.__version__, gpus=torch.cuda.device_count(),
                   gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

torch 2.11.0+cu128 | timm 1.0.30 | GPUs: ['Tesla T4', 'Tesla T4']
CPU cores: 4 | free disk /tmp: 1149 GB | /kaggle/working: 21 GB
previous output: None | finished runs carried over: 0


In [2]:
# ===================== CELL 2 — INDEX THE FACEFORENSICS++ FRAMES =====================
METHODS = {'deepfakes': 'DeepFakes', 'face2face': 'Face2Face', 'faceswap': 'FaceSwap',
           'neuraltextures': 'NeuralTextures', 'faceshifter': 'FaceShifter'}
REAL_KEYS = ('original', 'real', 'youtube', 'pristine')
IMG = ('.png', '.jpg', '.jpeg')

def find_ffpp():
    for root, dirs, _ in os.walk('/kaggle/input'):
        if root.count(os.sep) > 7: dirs[:] = []; continue
        low = [d.lower().replace('_', '').replace('-', '') for d in dirs]
        if sum(any(k == l or l.startswith(k) for l in low) for k in METHODS) >= 3:
            has_real = any(any(r in l for r in REAL_KEYS) for l in low)
            return root if has_real else os.path.dirname(root)   # e.g. manipulated_sequences/ next to original_sequences/
    return None

def classify(rel):
    parts = [p.lower().replace('_', '').replace('-', '') for p in rel.split('/')]
    if any('deepfakedetection' in p or 'actors' in p for p in parts): return None, None   # skip the DFD subset
    for i, p in enumerate(parts[:-1]):
        for k, v in METHODS.items():
            if p.startswith(k): return v, i
    for i, p in enumerate(parts[:-1]):
        if any(k in p for k in REAL_KEYS): return 'original', i
    low = rel.lower()                                     # method only in the file name
    for k, v in METHODS.items():
        if k in low.replace('_', ''): return v, len(parts) - 1
    return None, None

def video_id(rel, method, i):
    tail = '/'.join(rel.split('/')[i + 1:]) if i < len(rel.split('/')) - 1 else rel.split('/')[-1]
    if method == 'original':
        m = re.search(r'(?<!\d)(\d{3})(?!\d)', tail); return m.group(1) if m else None
    m = re.search(r'(?<!\d)(\d{3})_(\d{3})(?!\d)', tail); return f'{m.group(1)}_{m.group(2)}' if m else None

FFPP_DIR = FFPP_DIR or find_ffpp()
print('FF++ folder:', FFPP_DIR)
rows, unknown = [], []
for root, _, files in os.walk(FFPP_DIR or '/nonexistent'):
    for f in files:
        if not f.lower().endswith(IMG): continue
        p = os.path.join(root, f); rel = os.path.relpath(p, FFPP_DIR).replace(os.sep, '/')
        meth, i = classify(rel)
        vid = video_id(rel, meth, i) if meth else None
        (rows.append((p, 0 if meth == 'original' else 1, f'{meth}/{vid}', meth, vid)) if vid else unknown.append(rel))
idx = pd.DataFrame(rows, columns=['path', 'label', 'video', 'method', 'vid']).sort_values('path').reset_index(drop=True)
print('images indexed:', len(idx), '| not understood:', len(unknown))
if unknown: print('  examples not understood:', unknown[:5])
summary = idx.groupby('method').agg(frames=('path', 'size'), videos=('video', 'nunique'))
summary['frames_per_video'] = (summary.frames / summary.videos).round(1); print(summary.to_string())
for meth in summary.index: print(f'  sample {meth}:', os.path.relpath(idx[idx.method == meth].path.iloc[0], FFPP_DIR))
from PIL import Image
sizes = pd.Series([Image.open(p).size for p in idx.path.sample(min(200, len(idx)), random_state=0)]).value_counts().head(5)
print('image sizes (w,h) in a random sample of 200:', dict(sizes))
split_dirs = sorted({d for d in ('train', 'val', 'test') if idx.path.str.contains(f'/{d}/').any()})
print('dataset already has split folders:', split_dirs or 'no', '(ignored — we split by video id)')
missing_methods = [m for m in METHODS.values() if m not in summary.index]
if missing_methods: print('methods not found in this dataset:', missing_methods)
ok = ('original' in summary.index and summary.videos.get('original', 0) >= 900 and not missing_methods
      and all(summary.videos.get(m, 0) >= 900 for m in METHODS.values()) and len(unknown) < 0.01 * max(len(idx), 1))
print('STATUS:', 'OK' if ok else 'PROBLEM — paste this cell output to Claude')
DIAG['index'] = dict(folder=FFPP_DIR, images=len(idx), not_understood=len(unknown), summary=summary.reset_index().to_dict('records'),
                     image_sizes={str(k): int(v) for k, v in sizes.items()}, split_dirs=split_dirs); save_diag()

FF++ folder: /kaggle/input/datasets/adham7elmy/faceforencispp-extracted-frames
images indexed: 191550 | not understood: 0
                frames  videos  frames_per_video
method                                          
DeepFakes        31843     999              31.9
Face2Face        31949     999              32.0
FaceShifter      31924     999              32.0
FaceSwap         31954    1000              32.0
NeuralTextures   31931     999              32.0
original         31949     999              32.0
  sample DeepFakes: fake/Deepfakes/000_003/000.png
  sample Face2Face: fake/Face2Face/000_003/000.png
  sample FaceShifter: fake/FaceShifter/000_003/000.png
  sample FaceSwap: fake/FaceSwap/000_003/000.png
  sample NeuralTextures: fake/NeuralTextures/000_003/000.png
  sample original: real/000/000.png
image sizes (w,h) in a random sample of 200: {(256, 256): np.int64(200)}
dataset already has split folders: no (ignored — we split by video id)
STATUS: OK


In [3]:
# ===================== CELL 3 — OFFICIAL VIDEO-LEVEL SPLIT + PER-PARTITION BALANCING =====================
import urllib.request
splits, src = {}, 'official FF++ split files'
try:
    for p in ['train', 'val', 'test']:
        url = f'https://raw.githubusercontent.com/ondyari/FaceForensics/master/dataset/splits/{p}.json'
        splits[p] = json.loads(urllib.request.urlopen(url, timeout=30).read())
except Exception as e:
    src = f'FALLBACK seeded split (could not download official files: {e})'
    fakes = idx[idx.label == 1].vid.unique(); pairs = sorted({tuple(sorted(v.split('_'))) for v in fakes})
    random.Random(42).shuffle(pairs)
    splits = {'train': pairs[:360], 'val': pairs[360:430], 'test': pairs[430:500]}
part_of = {v: p for p, prs in splits.items() for pr in prs for v in pr}
def part(r):
    if r.label == 0: return part_of.get(r.vid)
    a, b = r.vid.split('_'); pa, pb = part_of.get(a), part_of.get(b)
    return pa if pa == pb else None
idx['partition'] = [part(r) for r in idx.itertuples()]
orphans = idx.partition.isna().sum(); idx = idx[idx.partition.notna()].copy()
if FRAMES_PER_VIDEO:
    idx = idx.groupby('video', group_keys=False).apply(
        lambda g: g.iloc[np.linspace(0, len(g) - 1, min(FRAMES_PER_VIDEO, len(g))).round().astype(int)])
rng = np.random.default_rng(42); keep = []
for p in ['train', 'val', 'test']:
    real = idx[(idx.partition == p) & (idx.label == 0)]; keep.append(real)
    n, ms = len(real), [m for m in METHODS.values() if (idx.method == m).any()]
    for k, m in enumerate(ms):
        pool = idx[(idx.partition == p) & (idx.method == m)]
        take = min(len(pool), n // len(ms) + (1 if k < n % len(ms) else 0))
        keep.append(pool.iloc[rng.choice(len(pool), take, replace=False)])
sv = pd.concat(keep).reset_index(drop=True)
assert sv.groupby('video').partition.nunique().max() == 1
src_ids = pd.concat([sv[['vid', 'partition']].assign(i=sv.vid.str.split('_')).explode('i')[['i', 'partition']]])
id_leak = (src_ids.groupby('i').partition.nunique() > 1).sum()
tab = sv.groupby('partition').apply(lambda d: pd.Series(dict(
    source_videos=d[d.label == 0].vid.nunique(), fake_videos=d[d.label == 1].video.nunique(),
    real_frames=(d.label == 0).sum(), fake_frames=(d.label == 1).sum(), total=len(d)))).loc[['train', 'val', 'test']]
tab.loc['total'] = tab.sum()
print('split source:', src); print('frames without a partition (dropped):', orphans)
print(tab.to_string())
print('fake frames per method per partition:'); print(sv[sv.label == 1].groupby(['partition', 'method']).size().unstack().to_string())
print('source ids appearing in more than one partition:', id_leak)
# old leaky protocol: random frame-level 80/10/10 on the same balanced pool (seed 42), for Section III-E only
from sklearn.model_selection import train_test_split
tr, rest = train_test_split(sv.index, test_size=0.2, stratify=sv.label, random_state=42)
va, te = train_test_split(rest, test_size=0.5, stratify=sv.label[rest], random_state=42)
sf = sv.copy(); sf.loc[tr, 'partition'] = 'train'; sf.loc[va, 'partition'] = 'val'; sf.loc[te, 'partition'] = 'test'
shared = set(sf[sf.partition == 'train'].video) & set(sf[sf.partition == 'test'].video)
frac = sf[sf.partition == 'test'].video.isin(shared).mean() * 100
print(f'frame-level split: {len(shared)} of {sf[sf.partition == "test"].video.nunique()} test videos also appear in training '
      f'({frac:.1f}% of test frames)')
ok = orphans < 0.05 * (len(idx) + orphans) and id_leak == 0 and 'FALLBACK' not in src
print('STATUS:', 'OK' if ok else 'PROBLEM — paste this cell output to Claude')
DIAG['split'] = dict(source=src, orphans=int(orphans), table=tab.reset_index().to_dict('records'), id_leak=int(id_leak),
                     frame_split_sizes=sf.partition.value_counts().to_dict(), frame_split_shared_videos=len(shared),
                     frame_split_test_videos=int(sf[sf.partition == 'test'].video.nunique()), frame_split_test_frac=round(frac, 1),
                     per_method=sv[sv.label == 1].groupby(['partition', 'method']).size().unstack().to_dict()); save_diag()

/tmp/ipykernel_24/673392349.py:35: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  tab = sv.groupby('partition').apply(lambda d: pd.Series(dict(


split source: official FF++ split files
frames without a partition (dropped): 0
           source_videos  fake_videos  real_frames  fake_frames  total
partition                                                             
train                719         3592        22993        22993  45986
val                  140          698         4477         4477   8954
test                 140          700         4479         4479   8958
total                999         4990        31949        31949  63898
fake frames per method per partition:
method     DeepFakes  Face2Face  FaceShifter  FaceSwap  NeuralTextures
partition                                                             
test             896        896          895       896             896
train           4599       4599         4598      4599            4598
val              896        896          895       895             895
source ids appearing in more than one partition: 0
frame-level split: 3308 of 3316 test videos also a

In [4]:
# ===================== CELL 4 — DECODE ALL FRAMES ONCE INTO A FAST CACHE =====================
from concurrent.futures import ThreadPoolExecutor
def load_img(p):
    try: return np.asarray(Image.open(p).convert('RGB').resize((CACHE_SIZE, CACHE_SIZE), Image.BILINEAR), dtype=np.uint8)
    except Exception: return None
paths = sv.path.tolist(); N = len(paths)
need = N * CACHE_SIZE * CACHE_SIZE * 3 / 1e9
print(f'caching {N} frames at {CACHE_SIZE}x{CACHE_SIZE}: {need:.1f} GB needed, {shutil.disk_usage("/tmp").free / 1e9:.0f} GB free')
mmf = f'{CACHE}/ffpp_{CACHE_SIZE}.u8'
t = time.time(); bad = 0
if not (os.path.exists(mmf) and os.path.getsize(mmf) == N * CACHE_SIZE * CACHE_SIZE * 3):
    mm = np.memmap(mmf, dtype=np.uint8, mode='w+', shape=(N, CACHE_SIZE, CACHE_SIZE, 3))
    with ThreadPoolExecutor(2 * os.cpu_count()) as pool:
        for i, a in enumerate(pool.map(load_img, paths)):
            if a is None: bad += 1
            else: mm[i] = a
    mm.flush(); del mm
sv['cache_idx'] = np.arange(N); sf['cache_idx'] = np.arange(N)
sv.to_csv(f'{WORK}/split_video.csv', index=False); sf.to_csv(f'{WORK}/split_frame.csv', index=False)
print(f'done in {(time.time() - t) / 60:.1f} min; unreadable images: {bad}')
print('STATUS:', 'OK' if bad < 0.001 * N else 'PROBLEM — paste this cell output to Claude')
DIAG['cache'] = dict(frames=N, size=CACHE_SIZE, unreadable=bad); save_diag()

caching 63898 frames at 256x256: 12.6 GB needed, 1149 GB free
done in 1.9 min; unreadable images: 0
STATUS: OK


In [5]:
# ===================== CELL 5 — CELEB-DF (v2): FACE CROPS FROM THE OFFICIAL TEST VIDEOS =====================
import cv2
def find_celebdf():
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'List_of_testing_videos.txt' in files: return root
    return None
CELEBDF_DIR = CELEBDF_DIR or find_celebdf(); FACES = f'{WORK}/celebdf_faces'
print('Celeb-DF folder:', CELEBDF_DIR)
if CELEBDF_DIR is None:
    print('PROBLEM: Celeb-DF (v2) not found — add it as input, or the cross-dataset table will stay empty')
else:
    lst = [l.split() for l in open(f'{CELEBDF_DIR}/List_of_testing_videos.txt') if l.strip()]
    all_mp4 = {os.path.relpath(p, CELEBDF_DIR).replace(os.sep, '/'): p for p in glob.glob(f'{CELEBDF_DIR}/**/*.mp4', recursive=True)}
    by_name = {}
    for rel, p in all_mp4.items(): by_name.setdefault('/'.join(rel.split('/')[-2:]), p)
    from facenet_pytorch import MTCNN
    mtcnn = MTCNN(keep_all=False, select_largest=True, post_process=False, device='cuda:0')
    rows, missing, noface = [], 0, 0; t = time.time()
    for lab, rel in lst:
        vid = rel.replace('.mp4', ''); out = f'{FACES}/{vid}'
        if os.path.isdir(out) and len(os.listdir(out)) > 0:
            rows += [(vid, 0 if lab == '1' else 1, f) for f in sorted(glob.glob(f'{out}/*.jpg'))]; continue
        p = all_mp4.get(rel) or by_name.get('/'.join(rel.split('/')[-2:]))
        if p is None: missing += 1; continue
        cap = cv2.VideoCapture(p); n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); frames = []
        for k in np.linspace(0, max(n - 1, 0), CELEBDF_FRAMES).round().astype(int):
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(k)); okf, fr = cap.read()
            if okf: frames.append(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB))
        cap.release(); os.makedirs(out, exist_ok=True)
        if not frames: missing += 1; continue
        boxes, _ = mtcnn.detect([Image.fromarray(f) for f in frames])
        for j, (f, b) in enumerate(zip(frames, boxes)):
            if b is None: noface += 1; continue
            x1, y1, x2, y2 = b[0]; cx, cy, s = (x1 + x2) / 2, (y1 + y2) / 2, max(x2 - x1, y2 - y1) * 1.3   # 1.3x square margin
            H, W = f.shape[:2]; x1, y1 = int(max(cx - s / 2, 0)), int(max(cy - s / 2, 0)); x2, y2 = int(min(cx + s / 2, W)), int(min(cy + s / 2, H))
            fp = f'{out}/{j:02d}.jpg'; Image.fromarray(f[y1:y2, x1:x2]).resize((CACHE_SIZE, CACHE_SIZE)).save(fp, quality=95)
            rows.append((vid, 0 if lab == '1' else 1, fp))
    cd = pd.DataFrame(rows, columns=['video', 'label', 'path'])
    assert len(cd) > 0, 'no Celeb-DF faces extracted'
    mm = np.memmap(f'{CACHE}/celebdf_{CACHE_SIZE}.u8', dtype=np.uint8, mode='w+', shape=(len(cd), CACHE_SIZE, CACHE_SIZE, 3))
    with ThreadPoolExecutor(2 * os.cpu_count()) as pool:
        for i, a in enumerate(pool.map(load_img, cd.path.tolist())): mm[i] = a
    mm.flush(); del mm; del mtcnn; torch.cuda.empty_cache()
    cd['cache_idx'] = np.arange(len(cd)); cd.to_csv(f'{WORK}/celebdf_index.csv', index=False)
    vc = cd.groupby('label').video.nunique()
    print(f'listed videos: {len(lst)} | not found: {missing} | frames without a face: {noface} | {(time.time() - t) / 60:.1f} min')
    print(f'videos used: real {vc.get(0, 0)}, fake {vc.get(1, 0)} | face crops: {len(cd)}')
    print('STATUS:', 'OK' if missing == 0 and len(cd) > 0.9 * len(lst) * CELEBDF_FRAMES else 'PROBLEM — paste this cell output to Claude')
    DIAG['celebdf'] = dict(listed=len(lst), missing=missing, noface=noface, real_videos=int(vc.get(0, 0)),
                           fake_videos=int(vc.get(1, 0)), crops=len(cd), frames_per_video=CELEBDF_FRAMES); save_diag()

Celeb-DF folder: /kaggle/input/datasets/reubensuju/celeb-df-v2
listed videos: 518 | not found: 0 | frames without a face: 4 | 9.3 min
videos used: real 178, fake 340 | face crops: 16572
STATUS: OK


In [6]:
%%writefile /kaggle/working/dsrx_worker.py
# ===================== CELL 6 — TRAINING WORKER (written to a file, then launched on each GPU) =====================
# DS-RxNet revision — training worker. One process per GPU; both processes share one run queue.
import argparse, json, os, time, math, random, traceback, gc
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

ap = argparse.ArgumentParser(); ap.add_argument('--config', required=True); ap.add_argument('--gpu', default='0')
A = ap.parse_args()
C = json.load(open(A.config))
WORK = C['WORK']; T0 = time.time()
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.backends.cudnn.benchmark = True
IM_MEAN, IM_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
INPUT = {'irv2': 299}            # every other model: 224
LR = {'vgg16': (1e-4, 1e-4), 'irv2': (1e-4, 1e-4), 'vit': (2e-5, 0.05)}   # (lr, weight decay); default 1e-4 / 1e-4
for d in ['preds', 'ckpt', 'history', 'done', 'locks', 'logs']: os.makedirs(f'{WORK}/{d}', exist_ok=True)


def log(*a):
    msg = f'[gpu{A.gpu} {time.strftime("%H:%M:%S")}] ' + ' '.join(str(x) for x in a)
    print(msg, flush=True)


def isize(m): return INPUT.get(m, 224)


# ------------------------------------------------------------------ models
class Renorm(nn.Module):
    """Input arrives ImageNet-normalised; convert to the statistics the backbone's own weights expect."""
    def __init__(s, mean, std):
        super().__init__()
        mi, si, mb, sb = [torch.tensor(v, dtype=torch.float32).view(1, 3, 1, 1) for v in (IM_MEAN, IM_STD, mean, std)]
        s.register_buffer('a', si / sb); s.register_buffer('b', (mi - mb) / sb)
    def forward(s, x): return x * s.a + s.b


def timm_stats(name):
    import timm
    cfg = timm.create_model(name, pretrained=False).pretrained_cfg
    return cfg.get('mean', IM_MEAN), cfg.get('std', IM_STD)


def freeze_except(m, keep):
    for n, p in m.named_parameters(): p.requires_grad = any(n.startswith(k) for k in keep)


def head(d_in):
    return nn.Sequential(nn.Linear(d_in, 1024), nn.BatchNorm1d(1024), nn.SiLU(),
                         nn.Linear(1024, 512), nn.Dropout(0.4), nn.Linear(512, 2))


def stream_resnet(pre):  # Stream A — layer3 + layer4 trainable
    import torchvision
    m = torchvision.models.resnet50(weights='IMAGENET1K_V1' if pre else None); m.fc = nn.Identity()
    freeze_except(m, ['layer3', 'layer4']); return m          # torchvision weights use ImageNet stats -> no renorm


def stream_xception(pre):  # Stream B — exit flow (block12, conv3, conv4) trainable
    import timm
    m = timm.create_model('legacy_xception', pretrained=pre, num_classes=0)
    freeze_except(m, ['block12', 'conv3', 'bn3', 'conv4', 'bn4'])
    return nn.Sequential(Renorm(*timm_stats('legacy_xception')), m)


class Single(nn.Module):
    def __init__(s, bb): super().__init__(); s.bb = bb; s.head = head(2048)
    def forward(s, x): return s.head(s.bb(x))


class DSRx(nn.Module):
    def __init__(s, fusion, pre):
        super().__init__(); s.a, s.b, s.fusion = stream_resnet(pre), stream_xception(pre), fusion
        if fusion == 'gated': s.gate = nn.Linear(4096, 2048)
        if fusion == 'attention': s.att = nn.Sequential(nn.Linear(4096, 256), nn.ReLU(), nn.Linear(256, 2))
        s.head = head(4096 if fusion == 'concat' else 2048)
    def forward(s, x):
        fa, fb = s.a(x), s.b(x); z = torch.cat([fa, fb], 1)              # Eq. (1)
        if s.fusion == 'sum': z = fa + fb                                   # Eq. (2)
        elif s.fusion == 'gated':                                           # Eq. (3)
            g = torch.sigmoid(s.gate(z)); z = g * fa + (1 - g) * fb
        elif s.fusion == 'attention':                                       # Eq. (4)
            w = torch.softmax(s.att(z), 1); z = w[:, :1] * fa + w[:, 1:] * fb
        return s.head(z)


class VGG(nn.Module):  # blocks 1-3 frozen, blocks 4-5 trainable, GAP -> 512 -> 2
    def __init__(s, pre):
        super().__init__(); import torchvision
        s.f = torchvision.models.vgg16(weights='IMAGENET1K_V1' if pre else None).features
        for i, l in enumerate(s.f):
            for p in l.parameters(): p.requires_grad = i >= 17
        s.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(512, 512), nn.BatchNorm1d(512),
                               nn.ReLU(), nn.Dropout(0.5), nn.Linear(512, 2))
    def forward(s, x): return s.head(s.f(x))


class IRv2(nn.Module):  # hypercolumn: Reduction-A stage (1088 ch) + final stage (1536 ch), 3x3 pooled
    def __init__(s, pre):
        super().__init__(); import timm
        s.rn = Renorm(*timm_stats('inception_resnet_v2'))
        s.bb = timm.create_model('inception_resnet_v2', pretrained=pre, features_only=True, out_indices=(3, 4))
        freeze_except(s.bb, ['repeat_1', 'mixed_7a', 'repeat_2', 'block8', 'conv2d_7b'])
        ch = s.bb.feature_info.channels()
        s.pool = nn.AdaptiveAvgPool2d(3)
        s.head = nn.Sequential(nn.Flatten(), nn.Linear(sum(ch) * 9, 512), nn.BatchNorm1d(512), nn.ReLU(),
                               nn.Dropout(0.5), nn.Linear(512, 2))
    def forward(s, x):
        f3, f4 = s.bb(s.rn(x)); return s.head(torch.cat([s.pool(f3), s.pool(f4)], 1))


class ViT(nn.Module):
    def __init__(s, pre):
        super().__init__(); import timm
        s.rn = Renorm(*timm_stats('vit_base_patch16_224'))
        s.bb = timm.create_model('vit_base_patch16_224', pretrained=pre, num_classes=0, attn_drop_rate=0.2)
        s.head = nn.Sequential(nn.Linear(768, 512), nn.GELU(), nn.Dropout(0.3), nn.Linear(512, 2))
    def forward(s, x): return s.head(s.bb(s.rn(x)))


def build(name, pre=True):
    if name.startswith('dsrxnet_'): return DSRx(name.split('_')[1], pre)
    return {'xception': lambda: Single(stream_xception(pre)), 'resnet50': lambda: Single(stream_resnet(pre)),
            'vgg16': lambda: VGG(pre), 'irv2': lambda: IRv2(pre), 'vit': lambda: ViT(pre)}[name]()


def freeze_bn_stats(model):  # BN layers inside frozen parts keep their ImageNet statistics
    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d) and not any(p.requires_grad for p in m.parameters()): m.eval()


# ------------------------------------------------------------------ data
class MM(Dataset):
    def __init__(s, mm_path, n, idx, labels, train, size):
        s.p, s.n, s.idx, s.y, s.train, s.size, s.mm = mm_path, n, np.asarray(idx), np.asarray(labels), train, size, None
        from torchvision.transforms import v2
        s.tf = v2.Compose([v2.RandomResizedCrop(size, scale=(0.8, 1.0), antialias=True), v2.RandomHorizontalFlip(),
                           v2.RandomRotation(10), v2.ColorJitter(0.2, 0.2, 0.2)])
    def __len__(s): return len(s.idx)
    def __getitem__(s, i):
        if s.mm is None: s.mm = np.memmap(s.p, dtype=np.uint8, mode='r', shape=(s.n, C['CACHE_SIZE'], C['CACHE_SIZE'], 3))
        x = torch.from_numpy(np.array(s.mm[s.idx[i]])).permute(2, 0, 1)
        x = s.tf(x) if s.train else F.interpolate(x[None].float(), size=(s.size, s.size), mode='bilinear',
                                                    antialias=True, align_corners=False)[0].round().clamp(0, 255).byte()
        return x, int(s.y[i])


def to_input(x):
    x = x.to(DEV, non_blocking=True).float() / 255.
    m = torch.tensor(IM_MEAN, device=DEV).view(1, 3, 1, 1); s = torch.tensor(IM_STD, device=DEV).view(1, 3, 1, 1)
    return (x - m) / s


def loader(ds, shuffle, seed):
    g = torch.Generator(); g.manual_seed(seed)
    return DataLoader(ds, C['BATCH'], shuffle=shuffle, drop_last=shuffle, num_workers=C['NUM_WORKERS'],
                      pin_memory=DEV == 'cuda', generator=g)


AMP = dict(device_type='cuda', dtype=torch.float16) if DEV == 'cuda' else dict(device_type='cpu', dtype=torch.bfloat16, enabled=False)


@torch.no_grad()
def predict(model, dl):
    model.eval(); out = []
    for x, _ in dl:
        with torch.autocast(**AMP): out.append(torch.softmax(model(to_input(x)).float(), 1)[:, 1].cpu())
    return torch.cat(out).numpy()


# ------------------------------------------------------------------ one run
def run(name):
    leak = name.startswith('leak_')
    model_name, seed = name.replace('leak_', '').rsplit('_seed', 1); seed = int(seed)
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    split = pd.read_csv(f"{WORK}/{'split_frame' if leak else 'split_video'}.csv")
    S = isize(model_name); mm = f"{C['CACHE']}/ffpp_{C['CACHE_SIZE']}.u8"
    part = {p: split[split.partition == p].reset_index(drop=True) for p in ['train', 'val', 'test']}
    dl = {p: loader(MM(mm, C['N_FFPP'], d.cache_idx, d.label, p == 'train', S), p == 'train', seed) for p, d in part.items()}
    model = build(model_name, C['PRETRAINED']).to(DEV)
    lr, wd = LR.get(model_name, (1e-4, 1e-4))
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=5, T_mult=2)
    scaler = torch.amp.GradScaler('cuda', enabled=DEV == 'cuda')
    crit = nn.CrossEntropyLoss(label_smoothing=0.1); crit_val = nn.CrossEntropyLoss()   # validation loss is UNSMOOTHED
    last, best_ck = f'{WORK}/ckpt/{name}_last.pt', f'{WORK}/ckpt/{name}_best.pt'
    st = dict(epoch=0, best=1e9, bad=0, hist=[], train_sec=0.0)
    if os.path.exists(last):  # resume an interrupted run (possibly from a previous Kaggle session)
        ck = torch.load(last, map_location=DEV, weights_only=False)
        model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt']); sched.load_state_dict(ck['sched'])
        scaler.load_state_dict(ck['scaler']); st = ck['st']; log(name, 'resumed at epoch', st['epoch'])
    nb = len(dl['train'])
    while st['epoch'] < C['MAX_EPOCHS'] and st['bad'] < C['PATIENCE']:
        ep = st['epoch']; t = time.time()
        epoch_est = max([h['sec'] for h in st['hist']], default=0)
        if (time.time() - T0) + 1.3 * epoch_est > C['TIME_BUDGET_H'] * 3600:
            log(name, 'time budget reached — paused, will resume next session'); return False
        model.train(); freeze_bn_stats(model); tl, n = 0.0, 0
        for it, (x, y) in enumerate(dl['train']):
            y = y.to(DEV)
            with torch.autocast(**AMP): loss = crit(model(to_input(x)), y)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            sched.step(ep + (it + 1) / nb); tl += loss.item() * len(y); n += len(y)
            if C.get('SMOKE') and it >= 2: break
        model.eval(); vl, vc, vn = 0.0, 0, 0
        with torch.no_grad():
            for x, y in dl['val']:
                y = y.to(DEV)
                with torch.autocast(**AMP): o = model(to_input(x)).float()
                vl += crit_val(o, y).item() * len(y); vc += (o.argmax(1) == y).sum().item(); vn += len(y)
                if C.get('SMOKE'): break
        vl /= vn; sec = time.time() - t
        st['hist'].append(dict(epoch=ep, train_loss=round(tl / n, 4), val_loss=round(vl, 4), val_acc=round(100 * vc / vn, 2),
                               sec=round(sec, 1), lr=opt.param_groups[0]['lr']))
        st['train_sec'] += sec; st['epoch'] = ep + 1
        if not math.isfinite(vl): log(name, 'NON-FINITE validation loss — stopping this run'); st['bad'] = C['PATIENCE']
        elif vl < st['best']: st['best'], st['bad'] = vl, 0; torch.save(model.state_dict(), best_ck)
        else: st['bad'] += 1
        torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(),
                        scaler=scaler.state_dict(), st=st), last)
        log(f'{name} ep{ep} train_loss {tl / n:.4f} val_loss {vl:.4f} val_acc {100 * vc / vn:.2f} ({sec / 60:.1f} min)')
    model.load_state_dict(torch.load(best_ck, map_location=DEV))
    te = part['test'].copy(); te['prob'] = predict(model, dl['test'])
    te[['path', 'video', 'method', 'label', 'prob']].to_csv(f'{WORK}/preds/{name}.csv', index=False)
    if not leak and os.path.exists(f'{WORK}/celebdf_index.csv'):
        cd = pd.read_csv(f'{WORK}/celebdf_index.csv')
        dcd = DataLoader(MM(f"{C['CACHE']}/celebdf_{C['CACHE_SIZE']}.u8", len(cd), cd.cache_idx, cd.label, False, S),
                         64, num_workers=C['NUM_WORKERS'])
        cd['prob'] = predict(model, dcd); cd[['video', 'label', 'prob']].to_csv(f'{WORK}/preds/celebdf_{name}.csv', index=False)
    json.dump(dict(st, run=name), open(f'{WORK}/history/{name}.json', 'w'))
    for f in (last, best_ck):
        if os.path.exists(f): os.remove(f)
    acc = 100 * ((te.prob >= .5) == te.label).mean()
    json.dump(dict(run=name, test_acc=round(acc, 2), epochs=st['epoch'], minutes=round(st['train_sec'] / 60, 1)),
              open(f'{WORK}/done/{name}.json', 'w'))
    log(f'DONE {name}: test acc {acc:.2f}% after {st["epoch"]} epochs, {st["train_sec"] / 60:.0f} min')
    return True


def claim(name):
    try: fd = os.open(f'{WORK}/locks/{name}.lock', os.O_CREAT | os.O_EXCL | os.O_WRONLY); os.close(fd); return True
    except FileExistsError: return False


for name in C['QUEUE']:
    if os.path.exists(f'{WORK}/done/{name}.json') or not claim(name): continue
    try:
        finished = run(name)
    except Exception:
        log('ERROR in', name); traceback.print_exc(); finished = None
        open(f'{WORK}/logs/ERROR_{name}.txt', 'w').write(traceback.format_exc())
    gc.collect(); torch.cuda.empty_cache()
    if finished is False: break   # time budget used up
log('worker finished')


Writing /kaggle/working/dsrx_worker.py


In [7]:
# ===================== CELL 7 — SMOKE TEST + PARAMETERS, GFLOPs, THROUGHPUT (Table X) =====================
QUEUE = ([f'{m}_seed{SEEDS[0]}' for m in ['dsrxnet_concat', 'xception', 'resnet50', 'vit', 'irv2', 'vgg16',
                                          'dsrxnet_sum', 'dsrxnet_gated', 'dsrxnet_attention']]
         + [f'leak_{m}_seed{SEEDS[0]}' for m in ['dsrxnet_concat', 'vit', 'irv2', 'vgg16']]
         + [f'{m}_seed{s}' for s in SEEDS[1:] for m in ['dsrxnet_concat', 'xception', 'resnet50', 'vit', 'irv2', 'vgg16',
                                                         'dsrxnet_sum', 'dsrxnet_gated', 'dsrxnet_attention']])
CFG = dict(WORK=WORK, CACHE=CACHE, CACHE_SIZE=CACHE_SIZE, N_FFPP=len(sv), BATCH=BATCH, MAX_EPOCHS=MAX_EPOCHS,
           PATIENCE=PATIENCE, TIME_BUDGET_H=TIME_BUDGET_H - (time.time() - globals().get('T_START', time.time())) / 3600,
           NUM_WORKERS=2, PRETRAINED=PRETRAINED, QUEUE=QUEUE)
json.dump(CFG, open(f'{WORK}/config.json', 'w'))
sys.argv = ['x', '--config', f'{WORK}/config.json']
import importlib.util
spec = importlib.util.spec_from_file_location('W', '/kaggle/working/dsrx_worker.py')
src_code = open('/kaggle/working/dsrx_worker.py').read().split('# ------------------------------------------------------------------ one run')[0]
W = type(sys)('W'); W.__dict__['__file__'] = 'w'; exec(compile(src_code, 'dsrx_worker', 'exec'), W.__dict__)
cost = []
for m in ['vgg16', 'irv2', 'vit', 'xception', 'resnet50', 'dsrxnet_concat', 'dsrxnet_sum', 'dsrxnet_gated', 'dsrxnet_attention']:
    net = None
    try:
        net = W.build(m, PRETRAINED).cuda(); S = W.isize(m)
        x = torch.randn(BATCH, 3, S, S, device='cuda'); y = torch.randint(0, 2, (BATCH,), device='cuda')
        net.train(); W.freeze_bn_stats(net)
        with torch.autocast('cuda', dtype=torch.float16): loss = torch.nn.functional.cross_entropy(net(x), y)
        loss.backward()
        tot = sum(p.numel() for p in net.parameters()) / 1e6; trn = sum(p.numel() for p in net.parameters() if p.requires_grad) / 1e6
        net.eval()
        try:
            from fvcore.nn import FlopCountAnalysis
            fa = FlopCountAnalysis(net, x[:1]); fa.unsupported_ops_warnings(False); fa.uncalled_modules_warnings(False)
            gf = fa.total() / 1e9
        except Exception: gf = float('nan')
        xb = torch.randn(64, 3, S, S, device='cuda')
        with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16):
            for _ in range(5): net(xb)
            torch.cuda.synchronize(); t = time.time()
            for _ in range(20): net(xb)
            torch.cuda.synchronize()
        cost.append(dict(model=m, params_M=round(tot, 1), trainable_M=round(trn, 1), GFLOPs=round(gf, 2),
                         img_per_s=round(64 * 20 / (time.time() - t)), status='OK'))
    except Exception as e:
        cost.append(dict(model=m, status=f'PROBLEM: {type(e).__name__}: {e}'))
    finally:
        del net; torch.cuda.empty_cache()
cost = pd.DataFrame(cost); print(cost.to_string(index=False))
print(f'runs in queue: {len(QUEUE)} | already finished: {sum(os.path.exists(f"{WORK}/done/{q}.json") for q in QUEUE)}')
print('STATUS:', 'OK' if (cost.status == 'OK').all() else 'PROBLEM — paste this cell output to Claude')
DIAG['cost'] = cost.to_dict('records'); DIAG['cost_gpu'] = torch.cuda.get_device_name(0); save_diag()

Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:02<00:00, 252MB/s]


model.safetensors: reconstructing file:   0%|          |  0.00B /  224MB            

model.safetensors: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

Downloading: "https://github.com/rwightman/pytorch-image-models/releases/download/v0.1-cadene/xception-43020ad28.pth" to /root/.cache/torch/hub/checkpoints/xception-43020ad28.pth
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 200MB/s]


            model  params_M  trainable_M  GFLOPs  img_per_s status
            vgg16      15.0         13.2   15.35        344     OK
             irv2      66.4         62.1   13.19        190     OK
              vit      86.2         86.2   16.87        334     OK
         xception      23.4          9.4    4.58        460     OK
         resnet50      26.1         24.7    4.11        704     OK
   dsrxnet_concat      49.0         33.6    8.69        266     OK
      dsrxnet_sum      46.9         31.5    8.69        256     OK
    dsrxnet_gated      55.3         39.9    8.69        257     OK
dsrxnet_attention      48.0         32.5    8.69        274     OK
runs in queue: 49 | already finished: 0
STATUS: OK


In [8]:
# ===================== CELL 8 — TRAIN: TWO WORKERS, ONE PER GPU (runs for hours; prints each finished run) =====================
procs = []
os.makedirs(f'{WORK}/logs', exist_ok=True)
for g in range(max(1, min(2, torch.cuda.device_count()))):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(g))
    procs.append(subprocess.Popen([sys.executable, '-u', '/kaggle/working/dsrx_worker.py', '--config', f'{WORK}/config.json', '--gpu', str(g)],
                                  env=env, stdout=open(f'{WORK}/logs/worker{g}.log', 'a'), stderr=subprocess.STDOUT))
    time.sleep(20)
seen = set(os.listdir(f'{WORK}/done')); t = time.time()
while any(p.poll() is None for p in procs):
    time.sleep(60)
    for f in sorted(set(os.listdir(f'{WORK}/done')) - seen):
        d = json.load(open(f'{WORK}/done/{f}')); seen.add(f)
        print(f"[{(time.time() - t) / 3600:.1f} h] finished {d['run']}: test acc {d['test_acc']}%, {d['epochs']} epochs, {d['minutes']} min", flush=True)
for f in sorted(set(os.listdir(f'{WORK}/done')) - seen):
    d = json.load(open(f'{WORK}/done/{f}')); print(f"finished {d['run']}: test acc {d['test_acc']}%, {d['epochs']} epochs, {d['minutes']} min")
for g in range(len(procs)):
    tail = open(f'{WORK}/logs/worker{g}.log').read().splitlines()[-4:]
    print(f'--- worker {g} last lines:', *tail, sep='\n   ')
errs = glob.glob(f'{WORK}/logs/ERROR_*.txt')
for e in errs: print('PROBLEM in', os.path.basename(e), '\n', open(e).read()[-1500:])
print('STATUS:', 'OK' if not errs else 'PROBLEM — paste this cell output to Claude')

[0.7 h] finished xception_seed0: test acc 75.14%, 13 epochs, 40.4 min
[1.0 h] finished dsrxnet_concat_seed0: test acc 86.98%, 13 epochs, 59.4 min
[1.4 h] finished resnet50_seed0: test acc 86.1%, 13 epochs, 38.0 min
[2.0 h] finished vit_seed0: test acc 88.32%, 7 epochs, 56.0 min
[2.4 h] finished irv2_seed0: test acc 86.83%, 7 epochs, 60.8 min
[2.5 h] finished vgg16_seed0: test acc 83.95%, 8 epochs, 27.6 min
[2.9 h] finished dsrxnet_gated_seed0: test acc 84.07%, 6 epochs, 26.5 min
[3.3 h] finished dsrxnet_sum_seed0: test acc 85.85%, 13 epochs, 52.8 min
[3.9 h] finished dsrxnet_attention_seed0: test acc 86.85%, 13 epochs, 56.4 min
[4.7 h] finished leak_dsrxnet_concat_seed0: test acc 98.06%, 19 epochs, 82.1 min
[5.2 h] finished leak_vit_seed0: test acc 97.73%, 9 epochs, 77.4 min
[6.1 h] finished leak_irv2_seed0: test acc 97.26%, 9 epochs, 84.8 min
[6.3 h] finished leak_vgg16_seed0: test acc 97.43%, 18 epochs, 66.0 min
[6.9 h] finished xception_seed1: test acc 75.14%, 10 epochs, 32.4 min
[7

In [9]:
# ===================== CELL 9 — REPORT (copy this whole output to Claude) =====================
import json, glob, os
import numpy as np, pandas as pd
from scipy.stats import norm, t as tdist, binomtest
from sklearn.metrics import roc_auc_score, roc_curve
D = json.load(open(f'{WORK}/diagnostics.json'))
MAIN = ['vgg16', 'irv2', 'vit', 'xception', 'resnet50', 'dsrxnet_concat']
ABL = ['xception', 'resnet50', 'average', 'dsrxnet_sum', 'dsrxnet_gated', 'dsrxnet_attention', 'dsrxnet_concat']
CMP = ['vgg16', 'irv2', 'vit', 'xception', 'resnet50', 'average']
QUEUE = json.load(open(f'{WORK}/config.json'))['QUEUE']

def seeds_done(m, pre=''):
    if m == 'average': return [s for s in SEEDS if all(os.path.exists(f'{WORK}/preds/{pre}{b}_seed{s}.csv') for b in ['xception', 'resnet50'])]
    return [s for s in SEEDS if os.path.exists(f'{WORK}/preds/{pre}{m}_seed{s}.csv')]
def load(m, s, pre=''):
    if m == 'average':
        a, b = load('xception', s, pre), load('resnet50', s, pre)
        if 'path' in a: assert (a.path.values == b.path.values).all()
        a = a.copy(); a['prob'] = (a.prob.values + b.prob.values) / 2; return a
    return pd.read_csv(f'{WORK}/preds/{pre}{m}_seed{s}.csv')
def vlevel(d): return d.groupby('video').agg(label=('label', 'first'), prob=('prob', 'mean')).reset_index()
def mets(y, p):
    q = (p >= .5).astype(int); TP = int(((q == 1) & (y == 1)).sum()); TN = int(((q == 0) & (y == 0)).sum())
    FP = int(((q == 1) & (y == 0)).sum()); FN = int(((q == 0) & (y == 1)).sum()); P = TP / max(TP + FP, 1); R = TP / max(TP + FN, 1)
    return dict(acc=100 * (TP + TN) / len(y), prec=100 * P, rec=100 * R, f1=100 * 2 * P * R / max(P + R, 1e-12), fnr=100 * FN / max(FN + TP, 1),
                fpr=100 * FP / max(FP + TN, 1), auc=roc_auc_score(y, p) if len(set(y)) > 1 else np.nan, TN=TN, FP=FP, FN=FN, TP=TP)
def fmt(xs, d=2):
    xs = np.array(xs, float)
    if len(xs) == 0: return '—'
    if len(xs) == 1: return f'{xs[0]:.{d}f} (n=1)'
    m, s = xs.mean(), xs.std(ddof=1); h = tdist.ppf(.975, len(xs) - 1) * s / np.sqrt(len(xs))
    return f'{m:.{d}f} ± {s:.{d}f} [{m - h:.{d}f}, {m + h:.{d}f}] (n={len(xs)})'
def mcn(c1, c2):
    b, c = int(np.sum(c1 & ~c2)), int(np.sum(~c1 & c2)); return b, c, (binomtest(b, b + c, .5).pvalue if b + c else 1.0)
def holm(ps):
    ps = np.asarray(ps, float); o = np.argsort(ps); out = np.empty(len(ps)); run = 0
    for r, i in enumerate(o): run = max(run, min(1, (len(ps) - r) * ps[i])); out[i] = run
    return out
def delong(y, p1, p2):
    def comp(p):
        P, N = p[y == 1], p[y == 0]
        v10 = np.array([(np.sum(x > N) + .5 * np.sum(x == N)) / len(N) for x in P])
        v01 = np.array([(np.sum(P > x) + .5 * np.sum(P == x)) / len(P) for x in N]); return v10.mean(), v10, v01
    a1, x1, y1 = comp(p1); a2, x2, y2 = comp(p2)
    S = np.cov(np.vstack([x1, x2])) / len(x1) + np.cov(np.vstack([y1, y2])) / len(y1)
    v = S[0, 0] + S[1, 1] - 2 * S[0, 1]; return 2 * norm.sf(abs(a1 - a2) / np.sqrt(v)) if v > 0 else 1.0

print('=' * 30, 'REPORT FOR CLAUDE — copy everything below', '=' * 30)
print('## ENV', D.get('env')); print('## INDEX', json.dumps(D.get('index'), default=str))
print('## SPLIT', json.dumps(D.get('split'), default=str)); print('## CACHE', D.get('cache')); print('## CELEBDF', D.get('celebdf'))
print('## COST (GPU %s, batch 64, AMP)' % D.get('cost_gpu'))
for r in D.get('cost', []): print('  ', r)
done = [q for q in QUEUE if os.path.exists(f'{WORK}/done/{q}.json')]
print(f'## RUNS {len(done)}/{len(QUEUE)} done.', 'ALL RUNS COMPLETE' if len(done) == len(QUEUE) else 'NOT COMPLETE — run another session')
print('   missing:', [q for q in QUEUE if q not in done])
for e in glob.glob(f'{WORK}/logs/ERROR_*.txt'): print('   ERROR', os.path.basename(e), open(e).read().strip().splitlines()[-1])

print('## TRAINING HISTORY per model, by seed (epochs run | best unsmoothed val loss @epoch | train minutes)')
for m in sorted({q.rsplit('_seed', 1)[0] for q in done}):
    items = []
    for q in [q for q in done if q.rsplit('_seed', 1)[0] == m]:
        h = json.load(open(f'{WORK}/history/{q}.json')); b = min(h['hist'], key=lambda r: r['val_loss'])
        items.append(f"s{q.rsplit('_seed', 1)[1]}:{h['epoch']}|{b['val_loss']}@{b['epoch']}|{h['train_sec'] / 60:.0f}m")
    print(f'   {m}:', '  '.join(items))
for q in ['dsrxnet_concat_seed0', 'irv2_seed0', 'vit_seed0']:
    if q in done: print(f'   curve {q} val_loss:', [r['val_loss'] for r in json.load(open(f"{WORK}/history/{q}.json"))['hist']])

R = {}
for m in sorted(set(MAIN + ABL)):
    R[m] = {}
    for s in seeds_done(m):
        d = load(m, s); r = mets(d.label.values, d.prob.values); v = vlevel(d); r['vauc'] = mets(v.label.values, v.prob.values)['auc']
        r['vacc'] = mets(v.label.values, v.prob.values)['acc']; R[m][s] = r
K = [('acc', 2), ('prec', 2), ('rec', 2), ('f1', 2), ('fnr', 2), ('fpr', 2), ('auc', 4), ('vauc', 4), ('vacc', 2)]
print('## TABLE V main (video-level FF++ test; mean ± SD [95% CI]) cols:', [k for k, _ in K])
for m in MAIN: print(f'   {m}:', ' | '.join(fmt([r[k] for r in R[m].values()], d) for k, d in K))
print('## TABLE VI confusion matrices, median-video-AUC seed (TN FP FN TP | real frames | fake frames)')
for m in MAIN:
    if not R[m]: continue
    ss = sorted(R[m], key=lambda s: R[m][s]['vauc']); s = ss[len(ss) // 2]; r = R[m][s]
    print(f"   {m} seed{s}: {r['TN']} {r['FP']} {r['FN']} {r['TP']} | {r['TN'] + r['FP']} | {r['FN'] + r['TP']}")
print('## TABLE VIII ablation cols:', [k for k, _ in K])
for m in ABL: print(f'   {m}:', ' | '.join(fmt([r[k] for r in R[m].values()], d) for k, d in K))
print('## PER-METHOD recall of manipulated frames (mean over seeds)')
for m in MAIN:
    ss = seeds_done(m)
    if ss:
        d = pd.concat([load(m, s) for s in ss]); f = d[d.label == 1]
        print(f'   {m}:', {k: round(100 * float((g.prob >= .5).mean()), 2) for k, g in f.groupby('method')})

print('## TABLE VII McNemar (DS-RxNet concat vs X): per seed b,c (frame) b,c (video); Holm-corrected max p over seeds; DeLong video AUC')
seeds_all = [s for s in seeds_done('dsrxnet_concat') if all(s in seeds_done(c) for c in CMP)]
P = {c: dict(f=[], v=[], d=[], bc=[]) for c in CMP}
for s in seeds_all:
    a = load('dsrxnet_concat', s); va = vlevel(a); pf, pv, pdl = [], [], []
    for c in CMP:
        b = load(c, s); assert (a.path.values == b.path.values).all(), 'TEST SETS DIFFER'; vb = vlevel(b)
        b1, c1, p1 = mcn((a.prob >= .5).values == a.label.values, (b.prob >= .5).values == b.label.values)
        b2, c2, p2 = mcn((va.prob >= .5).values == va.label.values, (vb.prob >= .5).values == vb.label.values)
        pf.append(p1); pv.append(p2); pdl.append(delong(va.label.values, va.prob.values, vb.prob.values)); P[c]['bc'].append((s, b1, c1, b2, c2))
    for c, x, y, z in zip(CMP, holm(pf), holm(pv), holm(pdl)): P[c]['f'].append(x); P[c]['v'].append(y); P[c]['d'].append(z)
for c in CMP:
    if P[c]['f']:
        print(f"   vs {c}: frame p_max={max(P[c]['f']):.3g} video p_max={max(P[c]['v']):.3g} DeLong p_max={max(P[c]['d']):.3g} | per seed (s,b_f,c_f,b_v,c_v): {P[c]['bc']}")
print('## DIVERSITY xception vs resnet50 (Yule Q, disagreement %)')
Q, Dg = [], []
for s in [s for s in seeds_done('xception') if s in seeds_done('resnet50')]:
    a, b = load('xception', s), load('resnet50', s); ca = (a.prob >= .5).values == a.label.values; cb = (b.prob >= .5).values == b.label.values
    n11, n00, n10, n01 = np.sum(ca & cb), np.sum(~ca & ~cb), np.sum(ca & ~cb), np.sum(~ca & cb)
    Q.append((n11 * n00 - n01 * n10) / max(n11 * n00 + n01 * n10, 1)); Dg.append(100 * (n01 + n10) / len(ca))
print('   Q =', fmt(Q, 3), '| disagreement =', fmt(Dg, 2))

print('## TABLE IX Celeb-DF v2 (frame AUC | video AUC | video acc@0.5 | FF++ video AUC)')
grid = np.linspace(0, 1, 21); ROC = {}
for m in MAIN + ['average']:
    fa, va_, vacc, tprs = [], [], [], []
    for s in SEEDS:
        if m == 'average':
            fs = [f'{WORK}/preds/celebdf_{b}_seed{s}.csv' for b in ['xception', 'resnet50']]
            if not all(os.path.exists(f) for f in fs): continue
            d = pd.read_csv(fs[0]); d['prob'] = (d.prob.values + pd.read_csv(fs[1]).prob.values) / 2
        else:
            f = f'{WORK}/preds/celebdf_{m}_seed{s}.csv'
            if not os.path.exists(f): continue
            d = pd.read_csv(f)
        fa.append(roc_auc_score(d.label, d.prob)); v = vlevel(d); va_.append(roc_auc_score(v.label, v.prob))
        vacc.append(100 * ((v.prob >= .5) == v.label).mean()); fp, tp, _ = roc_curve(v.label, v.prob); tprs.append(np.interp(grid, fp, tp))
    if fa:
        print(f'   {m}: {fmt(fa, 4)} | {fmt(va_, 4)} | {fmt(vacc, 2)} | {fmt([r["vauc"] for r in R.get(m, {}).values()], 4)}')
        ROC[m] = np.round(np.mean(tprs, 0), 3).tolist()
print('## ROC Celeb-DF video-level, mean TPR at FPR =', np.round(grid, 2).tolist())
for m, v in ROC.items(): print(f'   {m}:', v)

print('## LEAKAGE frame-level split (seed %d) vs video-level split' % SEEDS[0])
for m in ['vgg16', 'irv2', 'vit', 'dsrxnet_concat']:
    if seeds_done(m, 'leak_'):
        d = load(m, SEEDS[0], 'leak_'); r = mets(d.label.values, d.prob.values)
        vv = R[m].get(SEEDS[0])
        print(f"   {m}: frame-split acc {r['acc']:.2f} P {r['prec']:.2f} R {r['rec']:.2f} FNR {r['fnr']:.2f} FPR {r['fpr']:.2f} AUC {r['auc']:.4f} "
              f"(TN {r['TN']} FP {r['FP']} FN {r['FN']} TP {r['TP']}) | video-split acc seed{SEEDS[0]} {format(vv['acc'], '.2f') if vv else '—'}")
print('=' * 30, 'END OF REPORT', '=' * 30)


============================== REPORT FOR CLAUDE — copy everything below ==============================
## ENV {'torch': '2.11.0+cu128', 'timm': '1.0.30', 'gpus': 2, 'gpu': 'Tesla T4'}
## INDEX {"folder": "/kaggle/input/datasets/adham7elmy/faceforencispp-extracted-frames", "images": 191550, "not_understood": 0, "summary": [{"method": "DeepFakes", "frames": 31843, "videos": 999, "frames_per_video": 31.9}, {"method": "Face2Face", "frames": 31949, "videos": 999, "frames_per_video": 32.0}, {"method": "FaceShifter", "frames": 31924, "videos": 999, "frames_per_video": 32.0}, {"method": "FaceSwap", "frames": 31954, "videos": 1000, "frames_per_video": 32.0}, {"method": "NeuralTextures", "frames": 31931, "videos": 999, "frames_per_video": 32.0}, {"method": "original", "frames": 31949, "videos": 999, "frames_per_video": 32.0}], "image_sizes": {"(256, 256)": 200}, "split_dirs": []}
## SPLIT {"source": "official FF++ split files", "orphans": 0, "table": [{"partition": "train", "source_videos": 719